# Explore ViFactCheck Dataset & Label Generation

This notebook shows:
1. What the ViFactCheck HuggingFace dataset looks like (raw claims)
2. What our crawled data looks like (articles without labels)
3. How `recover_labels.py` matches URLs to inject labels
4. The 3 label mapping strategies and their distributions
5. Why the training set is imbalanced

In [1]:
import json
import os
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse, urlunparse

# Install datasets if needed
try:
    from datasets import load_dataset
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets"])
    from datasets import load_dataset

data_dir = Path("../data/json")
print(f"Data dir: {data_dir.resolve()}")

Data dir: G:\My Drive\Thesis_Final\fake-new-detection\notebooks\data\json


## 1. ViFactCheck HuggingFace Dataset (Raw)

The dataset `tranthaihoa/vifactcheck` contains Vietnamese fact-checking claims.
Each record is a **claim** about a news article, NOT the article itself.

In [3]:
# Load ViFactCheck from HuggingFace
hf_train = load_dataset("tranthaihoa/vifactcheck", split="train")
hf_dev = load_dataset("tranthaihoa/vifactcheck", split="dev")
hf_test = load_dataset("tranthaihoa/vifactcheck", split="test")

print(f"HuggingFace splits:")
print(f"  train:      {len(hf_train)} claims")
print(f"  validation: {len(hf_dev)} claims")
print(f"  test:       {len(hf_test)} claims")
print(f"\nColumns: {hf_train.column_names}")

HuggingFace splits:
  train:      5062 claims
  validation: 723 claims
  test:       1447 claims

Columns: ['Unnamed: 0', 'index', 'Statement', 'Context', 'annotation_id', 'Topic', 'Author', 'Url', 'labels', 'Evidence']


In [4]:
# Show sample records
print("=" * 60)
print("SAMPLE RECORDS FROM VIFACTCHECK")
print("=" * 60)
for i in range(3):
    item = hf_train[i]
    print(f"\n--- Record {i} ---")
    for key, val in item.items():
        val_str = str(val)
        if len(val_str) > 150:
            val_str = val_str[:150] + "..."
        print(f"  {key}: {val_str}")

SAMPLE RECORDS FROM VIFACTCHECK

--- Record 0 ---
  Unnamed: 0: 0
  index: 3049
  Statement: Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ, Thủ tướng Chính phủ chúc mừng Đài Truyền hình Việt Nam, Đài truyền hình các tỉnh, thành phố trên cả nướ...
  Context: (Chinhphu.vn) - Đây là mong muốn, gửi gắm của Phó Thủ tướng Trần Hồng Hà đến những người làm truyền hình tại lễ bế mạc Liên hoan Truyền hình toàn quốc...
  annotation_id: 18933775
  Topic: Chính trị
  Author: Chính Phủ
  Url: https://baochinhphu.vn/lan-toa-nhung-gia-tri-van-hoa-tot-dep-chan-thien-my-cua-dan-toc-102230318225051612.htm
  labels: 0
  Evidence: Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó Thủ tướng Trần Hồng Hà chúc mừng Đài Truyền hình Việt Nam, Đài truyền hình các tỉnh, thành phố trên cả nư...

--- Record 1 ---
  Unnamed: 0: 1
  index: 6811
  Statement: Hành vi của Tô Văn Hải là cho phép người khác đổ chất thải rắn ra môi trường vàchôn, lấp chất thải trái phép.
  Context: Ngày 24/3, Cơ quan Cảnh sát điều tra Công an t

In [5]:
# ViFactCheck label distribution
# Labels: 0=SUPPORTED (real), 1=REFUTED (fake), 2=NEI (not enough info)
LABEL_NAMES = {0: "SUPPORTED (Real)", 1: "REFUTED (Fake)", 2: "NEI (Not Enough Info)", None: "None (Missing)"}

print("VIFACTCHECK LABEL DISTRIBUTION (per split)")
print("=" * 60)
for name, ds in [("train", hf_train), ("validation", hf_dev), ("test", hf_test)]:
    labels = [item.get("labels") for item in ds]
    dist = Counter(labels)
    total = len(labels)
    print(f"\n{name} ({total} claims):")
    for lbl in sorted(dist, key=lambda x: (x is None, x)):
        pct = dist[lbl] / total * 100
        print(f"  {LABEL_NAMES.get(lbl, lbl)}: {dist[lbl]} ({pct:.1f}%)")

VIFACTCHECK LABEL DISTRIBUTION (per split)

train (5062 claims):
  SUPPORTED (Real): 1751 (34.6%)
  REFUTED (Fake): 1658 (32.8%)
  NEI (Not Enough Info): 1653 (32.7%)

validation (723 claims):
  SUPPORTED (Real): 256 (35.4%)
  REFUTED (Fake): 244 (33.7%)
  NEI (Not Enough Info): 223 (30.8%)

test (1447 claims):
  SUPPORTED (Real): 508 (35.1%)
  REFUTED (Fake): 468 (32.3%)
  NEI (Not Enough Info): 471 (32.6%)


## 2. Our Crawled Data (No Labels)

We crawled the actual news articles from the URLs in ViFactCheck.
These crawled files have article text + images but **NO labels** — that's the problem `recover_labels.py` solves.

In [6]:
# Show crawled data structure (raw, no labels)
raw_path = data_dir / "news_data_vifactcheck_train.json"
if raw_path.exists():
    with open(raw_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    
    print(f"Raw crawled train: {len(raw_data)} articles")
    print(f"Keys: {list(raw_data[0].keys())}")
    print(f"Has 'label' field: {'label' in raw_data[0]}")
    
    print(f"\n--- Sample crawled article ---")
    sample = raw_data[0]
    print(f"  title: {sample.get('title', 'N/A')[:100]}...")
    print(f"  content length: {len(sample.get('content', ''))} chars")
    print(f"  source_url: {sample.get('source_url', 'N/A')}")
    print(f"  images: {len(sample.get('images', []))} images")
    print(f"  label: {sample.get('label', 'MISSING')}")
else:
    print(f"Raw file not found: {raw_path}")

Raw crawled train: 4509 articles
Keys: ['title', 'content', 'source_url', 'other_urls', 'images', 'contents', 'markdown']
Has 'label' field: False

--- Sample crawled article ---
  title: Lan toả những giá trị văn hóa tốt đẹp 'chân - thiện - mỹ' của dân tộc...
  content length: 6984 chars
  source_url: https://baochinhphu.vn/lan-toa-nhung-gia-tri-van-hoa-tot-dep-chan-thien-my-cua-dan-toc-102230318225051612.htm
  images: 5 images
  label: MISSING


## 3. How Label Matching Works

`recover_labels.py` matches crawled articles to ViFactCheck claims by **URL**:

```
ViFactCheck claim (HuggingFace)     Crawled article (our JSON)
  Url: "https://vnexpress.net/..."    source_url: "https://vnexpress.net/..."
  labels: 1 (REFUTED)         -->     label: 1 (Fake)
```

When multiple claims share the same URL, the most severe label wins:
- REFUTED (1) > NEI (2) > SUPPORTED (0)

In [7]:
# Demonstrate URL matching
def normalize_url(url):
    """Same normalization as recover_labels.py"""
    if not url:
        return ""
    try:
        parsed = urlparse(url.strip())
        path = parsed.path.rstrip("/") if parsed.path != "/" else "/"
        return urlunparse((
            parsed.scheme.lower(), parsed.netloc.lower(),
            path, parsed.params, parsed.query, ""
        ))
    except Exception:
        return url.strip().rstrip("/")


# Build URL -> label map from HuggingFace
url_labels = {}  # url -> list of labels
for item in hf_train:
    url = item.get("Url", "")
    label = item.get("labels")
    if url and label is not None:
        norm = normalize_url(url)
        if norm not in url_labels:
            url_labels[norm] = []
        url_labels[norm].append(int(label))

# Show matching stats
n_urls = len(url_labels)
n_conflicts = sum(1 for v in url_labels.values() if len(set(v)) > 1)
print(f"Unique URLs in HF train: {n_urls}")
print(f"URLs with conflicting labels: {n_conflicts}")

# Show examples of conflicts
print(f"\n--- Example URL conflicts (multiple claims, same article) ---")
conflict_count = 0
for url, labels in url_labels.items():
    if len(set(labels)) > 1:
        label_names = [LABEL_NAMES.get(l, str(l)) for l in labels]
        print(f"  URL: {url[:80]}...")
        print(f"  Claims: {label_names}")
        severity = {1: 3, 2: 2, 0: 1}
        winner = max(set(labels), key=lambda x: severity.get(x, 0))
        print(f"  Resolved to: {LABEL_NAMES[winner]} (most severe wins)")
        print()
        conflict_count += 1
        if conflict_count >= 3:
            break

Unique URLs in HF train: 1035
URLs with conflicting labels: 963

--- Example URL conflicts (multiple claims, same article) ---
  URL: https://baochinhphu.vn/lan-toa-nhung-gia-tri-van-hoa-tot-dep-chan-thien-my-cua-d...
  Claims: ['SUPPORTED (Real)', 'REFUTED (Fake)', 'NEI (Not Enough Info)', 'NEI (Not Enough Info)', 'REFUTED (Fake)', 'SUPPORTED (Real)']
  Resolved to: REFUTED (Fake) (most severe wins)

  URL: https://baotintuc.vn/an-ninh-trat-tu/bat-tam-giam-doi-tuong-chon-lap-hon-600-tan...
  Claims: ['SUPPORTED (Real)', 'NEI (Not Enough Info)']
  Resolved to: NEI (Not Enough Info) (most severe wins)

  URL: https://plo.vn/9-quan-huyen-o-tphcm-se-bi-cup-nuoc-vao-cuoi-tuan-nay-post725015....
  Claims: ['REFUTED (Fake)', 'NEI (Not Enough Info)', 'SUPPORTED (Real)', 'NEI (Not Enough Info)', 'SUPPORTED (Real)']
  Resolved to: REFUTED (Fake) (most severe wins)



In [8]:
# Show how many crawled articles match HF URLs
if raw_path.exists():
    matched = 0
    unmatched = 0
    for article in raw_data:
        norm = normalize_url(article.get("source_url", ""))
        if norm in url_labels:
            matched += 1
        else:
            unmatched += 1
    
    print(f"Crawled articles:  {len(raw_data)}")
    print(f"  Matched to HF:   {matched} ({matched/len(raw_data)*100:.1f}%)")
    print(f"  No URL match:    {unmatched} ({unmatched/len(raw_data)*100:.1f}%)")
    print(f"\nUnmatched articles are DROPPED (no label = can't train)")

Crawled articles:  4509
  Matched to HF:   4509 (100.0%)
  No URL match:    0 (0.0%)

Unmatched articles are DROPPED (no label = can't train)


## 4. Three Label Mapping Strategies

ViFactCheck has 3 labels. We need binary (Real/Fake) for COOLANT.
Three strategies handle the NEI (Not Enough Info) class differently:

| Strategy | SUPPORTED | REFUTED | NEI | None |
|----------|-----------|---------|-----|------|
| `binary_exclude_nei` | Real (0) | Fake (1) | **dropped** | dropped |
| `binary_nei_as_real` | Real (0) | Fake (1) | **Real (0)** | dropped |
| `three_class` | Real (0) | Fake (1) | **NEI (2)** | dropped |

In [9]:
# Show label distribution for all 3 strategies
strategies = {
    "binary_exclude_nei": data_dir,
    "binary_nei_as_real": data_dir / "labeled_nei_as_real",
    "three_class": data_dir / "labeled_three_class",
}

BINARY_NAMES = {0: "Real", 1: "Fake", 2: "NEI"}

print("LABEL DISTRIBUTION BY STRATEGY")
print("=" * 70)

for strategy, dir_path in strategies.items():
    print(f"\n>>> Strategy: {strategy}")
    print(f"    Directory: {dir_path}")
    
    for split in ["train", "dev", "test"]:
        path = dir_path / f"news_data_vifactcheck_{split}_labeled.json"
        if not path.exists():
            print(f"    {split}: FILE NOT FOUND")
            continue
        
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        labels = [d["label"] for d in data]
        dist = Counter(labels)
        total = len(labels)
        parts = [f"{BINARY_NAMES.get(k, k)}={v} ({v/total*100:.1f}%)" for k, v in sorted(dist.items())]
        print(f"    {split:5s}: {total:5d} samples | {' | '.join(parts)}")

LABEL DISTRIBUTION BY STRATEGY

>>> Strategy: binary_exclude_nei
    Directory: ..\data\json
    train:  4259 samples | Real=91 (2.1%) | Fake=4168 (97.9%)
    dev  :   450 samples | Real=127 (28.2%) | Fake=323 (71.8%)
    test :   961 samples | Real=175 (18.2%) | Fake=786 (81.8%)

>>> Strategy: binary_nei_as_real
    Directory: ..\data\json\labeled_nei_as_real
    train:  4509 samples | Real=341 (7.6%) | Fake=4168 (92.4%)
    dev  :   634 samples | Real=311 (49.1%) | Fake=323 (50.9%)
    test :  1282 samples | Real=496 (38.7%) | Fake=786 (61.3%)

>>> Strategy: three_class
    Directory: ..\data\json\labeled_three_class
    train:  4509 samples | Real=91 (2.0%) | Fake=4168 (92.4%) | NEI=250 (5.5%)
    dev  :   634 samples | Real=127 (20.0%) | Fake=323 (50.9%) | NEI=184 (29.0%)
    test :  1282 samples | Real=175 (13.7%) | Fake=786 (61.3%) | NEI=321 (25.0%)


## 5. Why Train is Imbalanced

The imbalance comes from **ViFactCheck itself**, not from our processing.

In [ ]:
# Trace the imbalance back to the source
print("WHERE THE IMBALANCE COMES FROM")
print("=" * 60)

print("\n1. HuggingFace ViFactCheck (raw claims):")
for name, ds in [("train", hf_train), ("validation", hf_dev), ("test", hf_test)]:
    labels = [item.get("labels") for item in ds]
    dist = Counter(labels)
    total = len(labels)
    sup = dist.get(0, 0)
    ref = dist.get(1, 0)
    nei = dist.get(2, 0)
    none_ = dist.get(None, 0)
    print(f"  {name:12s}: SUP={sup} REF={ref} NEI={nei} None={none_} | REF/SUP ratio = {ref/max(sup,1):.1f}:1")

print("\n2. After URL matching (crawled -> labeled):")
print("  Many URLs map to REFUTED claims (fake news articles are fact-checked more)")
print("  SUPPORTED articles are rarer in fact-checking datasets")

print("\n3. With binary_nei_as_real strategy:")
nei_path = data_dir / "labeled_nei_as_real" / "news_data_vifactcheck_train_labeled.json"
if nei_path.exists():
    with open(nei_path, "r", encoding="utf-8") as f:
        train_nei = json.load(f)
    labels = [d["label"] for d in train_nei]
    dist = Counter(labels)
    real = dist.get(0, 0)
    fake = dist.get(1, 0)
    print(f"  Train: {real} Real + {fake} Fake = {real+fake} total")
    print(f"  Ratio: {fake/max(real,1):.1f}:1 (Fake:Real)")
    print(f"  NEI-as-real adds ~{real - 91} samples to Real class (vs {91} pure SUPPORTED)")

print("\n4. Key insight:")
print("  - Fact-checking datasets are ALWAYS imbalanced toward fake/refuted")
print("  - Real news isn't fact-checked as often -> fewer SUPPORTED claims")
print("  - We handle this with: class-weighted loss + weighted sampling")
print("  - Dev set is nearly balanced (49/51) -> reliable validation")

In [ ]:
# Visual summary
print("\n" + "=" * 60)
print("SUMMARY: Label Pipeline")
print("=" * 60)
print("""
ViFactCheck HuggingFace (tranthaihoa/vifactcheck)
  |  Columns: claim_text, evidence, Url, labels (0/1/2/None)
  |
  v
recover_labels.py --strategy binary_nei_as_real
  |  1. Load HF dataset per split
  |  2. Build URL -> label map (normalize URLs)
  |  3. Resolve conflicts: REFUTED > NEI > SUPPORTED
  |  4. Match crawled article source_url to HF URL
  |  5. Map: SUPPORTED->0, NEI->0, REFUTED->1
  |  6. Drop articles with no URL match
  |
  v
labeled_nei_as_real/news_data_vifactcheck_*_labeled.json
  Train: 263 Real + 3113 Fake (after preprocessing drops)
  Dev:   222 Real + 243 Fake  (nearly balanced!)
  Test:  varies
""")